# Which shelves will be empty next week?

Six learners ranked by useful store checks (10 per store).

Companion notebook for *The Retail Data Science Playbook* (Shaon Biswas, 2026). Brightwell's and all data here are fictional and generated with a fixed seed, so running this notebook reproduces the numbers printed in the book.

Outputs are written to `../results/` and `../figures/`.

In [ ]:
import sys; sys.path.insert(0, '../case_studies')
from common import *
import pandas as pd, time
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss
import xgboost as xgb
from catboost import CatBoostClassifier
rng=np.random.default_rng(42)
nS,nK,nW=30,150,40
cats=['ambient','chilled','frozen','bakery','household','beers & wines']
sku_cat=rng.integers(0,6,nK); sup=rng.integers(0,25,nK); sup_rel=rng.beta(8,1.5,25)
rows=[]
for w in range(nW):
    promo=rng.random(nK)<0.08
    for s in range(nS):
        n=nK
        cover=rng.gamma(3,2.2,n)                      # days of stock cover
        lead=rng.choice([1,2,3,5],n,p=[.4,.3,.2,.1])
        fill=np.clip(sup_rel[sup]+rng.normal(0,.04,n),.5,1)
        cv=rng.gamma(2,.15,n)
        shelf=rng.uniform(.4,1.6,n)
        z=(-2.6 -0.55*np.log1p(cover)*2 + 1.8*promo*(cover<6) + 4*(0.95-fill) + 0.35*lead*(cover<lead*2)
           + 1.6*cv + 0.8*(shelf<0.6) + np.array([0,.3,.2,.6,-.3,0])[sku_cat] + rng.normal(0,.6,n))
        p=1/(1+np.exp(-z)); y=rng.random(n)<p
        rows.append(pd.DataFrame(dict(week=w,store=s,sku=np.arange(nK),category=np.array(cats)[sku_cat],supplier=sup.astype(str),
            cover_days=cover,lead_time=lead,fill_rate=fill,demand_cv=cv,shelf_ratio=shelf,promo_next=promo.astype(int),oos=y.astype(int))))
df=pd.concat(rows,ignore_index=True)
num=['cover_days','lead_time','fill_rate','demand_cv','shelf_ratio','promo_next']
X=pd.get_dummies(df[num+['category']],columns=['category'],dtype=float)
tr=df.week<30; va=(df.week>=30)&(df.week<34); te=df.week>=34
y=df.oos.values
res={}; preds={}
def ev(name,p,t0):
    pt=p[te.values]; yt=y[te.values]
    # business: Aisha's team can check 300 store-lines per week -> precision in the top 300 per week
    # constraint: each store can check its 10 highest-ranked lines per week (300 checks across 30 stores)
    d=df[te].assign(p=pt); hits=[g.nlargest(10,'p').oos.mean() for _,g in d.groupby(['week','store'])]
    res[name]=dict(pr_auc=average_precision_score(yt,pt),roc_auc=roc_auc_score(yt,pt),brier=brier_score_loss(yt,pt),hit300=np.mean(hits),secs=time.time()-t0); preds[name]=pt
t=time.time();m=make_pipeline(StandardScaler(),LogisticRegression(max_iter=500)).fit(X[tr],y[tr]);ev('Logistic regression',m.predict_proba(X)[:,1],t)
t=time.time();m=DecisionTreeClassifier(max_depth=4,min_samples_leaf=200).fit(X[tr],y[tr]);ev('Decision tree (depth 4)',m.predict_proba(X)[:,1],t)
t=time.time();sub=np.where(tr)[0];sub=rng.choice(sub,20000,replace=False);m=make_pipeline(StandardScaler(),KNeighborsClassifier(n_neighbors=50)).fit(X.iloc[sub],y[sub]);ev('k-NN (k = 50)',m.predict_proba(X)[:,1],t)
t=time.time();m=RandomForestClassifier(n_estimators=300,min_samples_leaf=20,max_features='sqrt',n_jobs=-1,random_state=0).fit(X[tr],y[tr]);ev('Random forest',m.predict_proba(X)[:,1],t);rf=m
t=time.time();m=xgb.XGBClassifier(n_estimators=2000,learning_rate=0.05,max_depth=5,subsample=.8,colsample_bytree=.8,early_stopping_rounds=100,eval_metric='aucpr',n_jobs=8)
m.fit(X[tr],y[tr],eval_set=[(X[va],y[va])],verbose=False);ev('XGBoost',m.predict_proba(X)[:,1],t);xbest=m.best_iteration
Xc=df[num+['category','supplier']]
t=time.time();m=CatBoostClassifier(iterations=2000,learning_rate=.05,depth=6,early_stopping_rounds=100,verbose=False,cat_features=['category','supplier'],thread_count=8)
m.fit(Xc[tr],y[tr],eval_set=(Xc[va],y[va]));ev('CatBoost',m.predict_proba(Xc)[:,1],t);cb=m
base=df[te].oos.mean()
# random baseline: hit rate = base rate ; rule baseline: cover_days < 3
d=df[te].assign(p=-df[te].cover_days);rule=np.mean([g.nlargest(10,'p').oos.mean() for _,g in d.groupby(['week','store'])])
fi=pd.Series(cb.get_feature_importance(),index=Xc.columns).sort_values(ascending=False)
out=dict(rows=len(df),train_rows=int(tr.sum()),test_rows=int(te.sum()),base_rate=base,rule_hit300=rule,xgb_best_iter=int(xbest),models=res,cb_importance=fi.round(1).to_dict())
dump('cs7',out)
# figure: hit rate at capacity
names=list(res);vals=[res[k]['hit300'] for k in names]
f,a=fig();order=np.argsort(vals);a.barh([names[i] for i in order]+[],[vals[i]*100 for i in order],color=[G if vals[i]==max(vals) else GR for i in order],height=.6)
a.axvline(rule*100,color=A,ls='--',lw=1.2);a.text(rule*100+.5,1.5,f'current rule\n(lowest cover): {rule*100:.0f}%',color=A,fontsize=6.5)
a.axvline(base*100,color=M,ls=':',lw=1);a.text(base*100+.5,3.5,f'random\n{base*100:.0f}%',color=M,fontsize=6.5)
a.set_xlabel('Share of each store\'s 10 weekly checks that find a real out-of-stock (%)');a.set_title('Which model makes the stores\' 10 checks count?');save(f,'cs7_hit')
print(json.dumps(out,indent=1,default=float)[:1500])


## Results used in the book

In [ ]:
import json, pprint
pprint.pprint(json.load(open('../results/cs7.json')), width=110, compact=True)

In [ ]:
from IPython.display import SVG, display
display(SVG('../figures/cs7_hit.svg'))